# imports

In [1]:
%reload_ext autoreload
%autoreload 2

import torch
import os
import json
from tqdm import tqdm
import numpy as np
import scipy.stats as stats

In [2]:
import pandas as pd

df = pd.read_csv('../results/accs_imagenet-1k_model_eval_350_sum_42.csv')
df.head()


,Unnamed: 0,GT,Optimal_global,Optimal_sample,Uniform_global_10,Uniform_sample_10,Random_global_10,Random_sample_10,Uniform_global_50,Uniform_sample_50,Random_global_50,Random_sample_50,Uniform_global_100,Uniform_sample_100,Random_global_100,Random_sample_100,Uniform_global_1000,Uniform_sample_1000,Random_global_1000,Random_sample_1000
0,0,82.646,82.646,84.410,85.0,65.0,91.522,69.412,87.0,87.0,80.324,78.342,82.5,81.5,78.246,77.856,83.25,88.15,81.438,83.578
1,1,76.476,76.476,76.802,85.0,65.0,91.522,69.412,71.0,69.0,78.858,74.756,77.5,68.5,77.856,80.606,76.45,77.95,76.772,77.748
2,2,80.310,80.310,80.776,85.0,75.0,91.522,69.412,79.0,81.0,83.362,80.004,81.5,82.5,78.246,76.276,79.65,79.75,79.880,80.646
3,3,0.120,0.120,0.002,5.0,5.0,15.600,15.600,1.0,1.0,0.298,0.298,0.5,0.5,0.178,0.178,0.05,0.05,0.148,0.148
4,4,78.020,78.020,79.018,75.0,55.0,69.412,60.756,81.0,81.0,83.362,86.168,80.5,79.5,77.958,75.172,78.25,81.35,77.034,82.080


In [3]:
# gt_scores = []
# accs_per_method = {}
# for row in df.iterrows():
#     print(row)
#     gt_scores.append(row['GT'])
#     for key, value in row.items():
#         if key != 'GT':
#             accs_per_method[key].append(value)
    # rank_corrs = []
    # # pred_accs_as_np = np.stack(list(predicted_accs.values()), axis=0)
    # pred_accs_as_np = np.stack(list(accs.values()), axis=0)
    # maes = np.abs(
    #     pred_accs_as_np - gt_scores[0][:, None]
    # )
    # for i in range(pred_accs_as_np.shape[1]):
    #     rank_corrs.append(safe_spearmanr(
    #         pred_accs_as_np[:, i],
    #         gt_scores[0][:, None],
    #     ))
    # maes_per_method[method_name] = maes
    # rank_corrs_per_method[method_name] = np.array(rank_corrs)

accs_per_method = {}
for column in df.columns:
    accs_per_method[column] = df[column]

In [4]:
STD_EPS = 1e-9


def safe_spearmanr(x_data, y_data):
    """
    Compute Spearman correlation with safe handling of constant arrays.

    Args:
        x_data: First array for correlation
        y_data: Second array for correlation

    Returns:
        float: Spearman correlation coefficient, or np.nan if either array is constant
    """
    # Check if either array is constant (all values are the same)
    if np.std(x_data) < STD_EPS or np.std(y_data) < STD_EPS:
        return np.nan  # or 0, depending on your preference
    else:
        return stats.spearmanr(x_data, y_data).statistic

In [ ]:
accs_per_method

In [5]:
gt_scores = np.array(list(accs_per_method['GT']))
print(gt_scores)
maes_per_method = {}
rank_corrs_per_method = {}
for method_name, accs in accs_per_method.items():
    if method_name in ['GT', 'Unnamed: 0']:
        continue
    rank_corrs = []
    # pred_accs_as_np = np.stack(list(predicted_accs.values()), axis=0)
    pred_accs_as_np = np.stack(list(accs), axis=0)
    maes = np.abs(
        pred_accs_as_np - gt_scores
    )
    # for i in range(pred_accs_as_np):
    rank_corrs.append(safe_spearmanr(
        pred_accs_as_np,
        gt_scores,
    ))
    maes_per_method[method_name] = maes
    rank_corrs_per_method[method_name] = np.array(rank_corrs)

[82.646 76.476 80.31   0.12  78.02  77.896 80.864 82.27  80.914 80.548
 46.592 78.712 82.97  78.742 83.122 86.804 79.41  78.252 83.436 84.894
 83.71  80.99  81.702 84.562 82.09  79.908 78.962  0.564 39.008 82.79
 76.384 78.462 78.616 79.732 78.416 83.96  83.526 84.226 83.436 83.154
 84.442 82.608 80.608 76.704 85.284 85.39  83.114 79.9   85.732 78.158]


In [6]:
for method_name, maes in maes_per_method.items():
    print(method_name, maes.mean())

Optimal_global 0.0
Optimal_sample 1.5639599999999998
Uniform_global_10 6.953600000000002
Uniform_sample_10 8.090880000000002
Random_global_10 9.627720000000005
Random_sample_10 12.162600000000005
Uniform_global_50 2.8667200000000013
Uniform_sample_50 3.8391200000000016
Random_global_50 2.348600000000002
Random_sample_50 4.058960000000002
Uniform_global_100 2.0028000000000006
Uniform_sample_100 2.7045600000000003
Random_global_100 2.0610800000000027
Random_sample_100 3.444840000000001
Uniform_global_1000 0.533760000000002
Uniform_sample_1000 2.3759200000000007
Random_global_1000 0.66424
Random_sample_1000 1.9059199999999998


In [7]:
rank_corrs_per_method

{'Optimal_global': array([1.]),
 'Optimal_sample': array([0.97726236]),
 'Uniform_global_10': array([0.75235276]),
 'Uniform_sample_10': array([0.72077099]),
 'Random_global_10': array([0.60874587]),
 'Random_sample_10': array([0.56389841]),
 'Uniform_global_50': array([0.85646007]),
 'Uniform_sample_50': array([0.80675848]),
 'Random_global_50': array([0.58224306]),
 'Random_sample_50': array([0.32485443]),
 'Uniform_global_100': array([0.71232917]),
 'Uniform_sample_100': array([0.57419794]),
 'Random_global_100': array([0.83826012]),
 'Random_sample_100': array([0.70241773]),
 'Uniform_global_1000': array([0.98124446]),
 'Uniform_sample_1000': array([0.87919778]),
 'Random_global_1000': array([0.98587698]),
 'Random_sample_1000': array([0.85904367])}